In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import math

class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes
        self.weights = nn.Parameter(torch.rand(in_channels, out_channels, modes, dtype=torch.cfloat))

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft(x)
        out_ft = torch.zeros(batchsize, self.out_channels, x.shape[-1]//2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes] = torch.einsum("bix,iox->box", x_ft[:, :, :self.modes], self.weights)
        x = torch.fft.irfft(out_ft, n=x.shape[-1])
        return x

class FNO1d(nn.Module):
    def __init__(self, modes=16, width=64):
        super().__init__()
        self.p = nn.Linear(1, width) 
        self.conv0 = SpectralConv1d(width, width, modes)
        self.mlp0 = nn.Linear(width, width)
        self.conv1 = SpectralConv1d(width, width, modes)
        self.mlp1 = nn.Linear(width, width)
        self.q = nn.Linear(width, 1) 

    def forward(self, x):
        x = self.p(x).permute(0, 2, 1) 
        x1 = self.conv0(x)
        x2 = self.mlp0(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = F.gelu(x1 + x2)
        x1 = self.conv1(x)
        x2 = self.mlp1(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = F.gelu(x1 + x2)
        x = self.q(x.permute(0, 2, 1))
        return x

def generate_heat_dataset(num_samples=1000, nx=100, alpha=0.01):
    ks = (torch.rand((num_samples, 1)) * 9.0) + 1.0 
    x = torch.linspace(0, 1, nx)
    inputs = torch.sin(ks * math.pi * x)
    decay = torch.exp(-alpha * (ks * math.pi)**2 * 0.01)
    outputs = inputs * decay
    return inputs.unsqueeze(-1), outputs.unsqueeze(-1)

def create_fno_factory(epochs=100, num_samples=2000, batch_size=64):
    print("--- Starting Amortized FNO Training Phase ---")
    train_start = time.time()
    inputs, outputs = generate_heat_dataset(num_samples=num_samples)
    dataset = data.TensorDataset(inputs, outputs)
    loader = data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = FNO1d(modes=16, width=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for x_batch, y_batch in loader:
            optimizer.zero_grad()
            pred = model(x_batch)
            loss = F.mse_loss(pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
    train_time = time.time() - train_start
    print(f"--- FNO Training Complete in {train_time:.2f} seconds ---")
    
    model.eval()
    def fno_solver_factory(k):
        x_test = torch.linspace(0, 1, 100)
        u_ic = torch.sin(k * math.pi * x_test)
        u_ic = u_ic.unsqueeze(0).unsqueeze(-1) 
        with torch.no_grad():
            u_pred = model(u_ic)
        u_pred_tensor = u_pred.squeeze(0)
        info_dict = {
            'loss': total_loss/len(loader),
            'amortized_train_time_sec': train_time,
            'solver_type': 'FNO'
        }
        return u_pred_tensor, info_dict
    return fno_solver_factory